In [ ]:
import pandas as pd
# 调用函数，不仅会生成 PDF，还会把数据返回给你
my_data = pd.read_pickle('/Volumes/genzelneuron/ephys_rec_data_20210612/log_20210612_Rat5_all_plot_data.pkl')
# 现在你可以随意使用这个 dataframe
my_data

,trial_ids,raw_x_scaled,raw_y_scaled,speed_raw_smoothed,speed_0_5s,speed_1_0s,speed_2_0s,speed_5_0s,time_seconds,normalized_time,stitched_time_seconds,physical_score_val,hops_score_val,path_physical_segments,path_topological_segments,node_sequence_str
0,"[1, 2, 3, 4, 5, 6, 7]","[[3.0612244897959187, 3.0688775510204085, 3.06...","[[0.8286516853932584, 0.8286516853932584, 0.82...","[[0.17219387755102122, 0.16262755102040893, 0....","[[0.16071428571428648, 0.1454081632653068, 0.1...","[[0.1683673469387763, 0.1645408163265314, 0.16...","[[0.2174562609942516, 0.2174562609942516, 0.22...","[[0.2100852815138141, 0.2085546692689161, 0.20...","[[0.0, 0.03333333333333333, 0.0666666666666666...","[[0.0, 5.557099194220617e-05, 0.00011114198388...","[[32.37712073567708, 32.410962822469074, 32.44...","[-2.530427661872706, -3.0447270028424436, -1.5...","[-1.9694406464655074, -2.630088659632498, -1.2...","[[[[397.0, 120.0], [370.0, 162.0]], [[370.0, 1...","[[[[397.0, 120.0], [451.0, 116.0]], [[451.0, 1...","[320,314,313,306,307,301,302,303,304,305,304,3..."


In [2]:
import spikeinterface.extractors as se
import spikeinterface as si
from spikeinterface.widgets import plot_probe_map
import spikeinterface as si  # import core only
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.postprocessing as spost
import spikeinterface.qualitymetrics as sqm
import spikeinterface.comparison as sc
import spikeinterface.exporters as sexp
import spikeinterface.curation as scur
import spikeinterface.widgets as sw
# Path to your Trodes .rec file
rec_path = r"/Volumes/genzelneuron/ephys_rec_data_20210612/Rat_Hm_Ephys_Rat5_406576_20210612_maze_merged.rec"

# Load it with the SpikeGadgets extractor
recording = se.read_spikegadgets(rec_path,all_annotations=True,use_names_as_ids=True)

print(recording)
print(recording.get_num_channels(), "channels")
print(recording.get_num_frames(), "samples")


/opt/miniconda3/envs/nwb4sachi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SpikeGadgetsRecordingExtractor: 128 channels - 30.0kHz - 1 segments - 125,285,321 samples 
                                4,176.18s (1.16 hours) - int16 dtype - 29.87 GiB
  file_path: /Volumes/genzelneuron/ephys_rec_data_20210612/Rat_Hm_Ephys_Rat5_406576_20210612_maze_merged.rec
128 channels
125285321 samples


In [4]:
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface as si
import matplotlib.pyplot as plt
import scipy.signal as signal
import numpy as np
from tqdm import tqdm  # Progress bar

# --- Load Data ---
rec_path = r"/Volumes/genzelneuron/ephys_rec_data_20210612/Rat_Hm_Ephys_Rat5_406576_20210612_maze_merged.rec"
recording = se.read_spikegadgets(rec_path, all_annotations=True, use_names_as_ids=True)

print(f"Original Sampling Rate: {recording.get_sampling_frequency()} Hz")

# --- Preprocessing ---

# 1. Bandpass Filter
recording = spre.bandpass_filter(recording, freq_min=1, freq_max=475)

# 2. Downsample to 1250 Hz (LFP)
target_fs = 1250
print(f"Resampling to {target_fs} Hz...")
recording_lfp = spre.resample(recording, target_fs)

# 3. Common Average Reference (CAR)
# reference='global' calculates the average of all channels
# operator='minus' subtracts this average from every channel
print("Applying Common Average Reference (CAR)...")
recording_car = spre.common_reference(recording_lfp, reference='global', operator='average')

# --- Data Loading ---

# Get Channel IDs
channel_ids = recording_car.get_channel_ids()
num_channels = len(channel_ids)

# Load data into memory
# Note: Since CAR is lazy, the actual subtraction happens here when we request the traces
print("Loading traces into memory (computing CAR on the fly)...")
traces = recording_car.get_traces(return_scaled=True)
print("Data loaded successfully.")

# --- PSD Calculation & Plotting ---

# Setup Plot Layout
n_cols = 8 
n_rows = int(np.ceil(num_channels / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 3 * n_rows), constrained_layout=True)
axes = axes.flatten() 

# Iterate with Progress Bar
for i, channel_id in enumerate(tqdm(channel_ids, desc="Calculating PSD")):
    ax = axes[i]
    
    # Get data for this channel
    data = traces[:, i]
    
    # Compute Power Spectrum (Welch's method)
    # nperseg determines frequency resolution
    freqs, psd = signal.welch(data, fs=target_fs, nperseg=2048)
    
    # Plotting (0 - 300 Hz)
    mask = freqs <= 300
    ax.plot(freqs[mask], psd[mask], color='b', lw=1)
    
    # Styling
    ax.set_title(f"Ch ID: {channel_id}", fontsize=10)
    ax.set_xlabel('Frequency (Hz)', fontsize=8)
    ax.set_ylabel('Power (uV^2/Hz)', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Power Spectra per Channel (LFP {int(target_fs)} Hz) - CAR Applied', fontsize=16)

print("Plotting complete. Showing window...")
plt.show()

Original Sampling Rate: 30000.0 Hz
Resampling to 1250 Hz...
Applying Common Average Reference (CAR)...
Loading traces into memory (computing CAR on the fly)...


: 